In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [3]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
import json
import re
import heapq
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Set, Tuple, Optional

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from dotenv import load_dotenv

import faiss

In [5]:
from datasets import load_dataset

# Adjust the slice size as needed: validation[:100] for quick testing, validation for full
dataset = load_dataset("rajpurkar/squad_v2", split="validation[:100]")


In [6]:

questions = []
ground_truths = []
all_contexts = []

for row in dataset:
    questions.append(row["question"])
    all_contexts.append(row["context"])
    if len(row["answers"]["text"]):
        ground_truths.append(row["answers"]["text"][0])
    else:
        ground_truths.append("")

print(f"Loaded {len(questions)} questions, {len(set(all_contexts))} unique contexts")

Loaded 100 questions, 17 unique contexts


In [7]:
# Deduplicate contexts and build a single document corpus
unique_contexts = list(dict.fromkeys(all_contexts))
doc_text = "\n".join(unique_contexts)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_text(doc_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 21


In [8]:
@dataclass
class Triple:
    head: str
    relation: str
    tail: str
    chunk_idx: int  # index of the source chunk

class KnowledgeGraph:
    """
    Knowledge Graph storing triplets with chunk associations.
    G = {(h, r, t, c) | c in D} as per Eq. (1) in the paper.
    """
    def __init__(self):
        self.triples: List[Triple] = []
        # Entity -> list of triple indices for fast lookup
        self.entity_index: Dict[str, List[int]] = defaultdict(list)
        # Chunk -> list of triple indices
        self.chunk_index: Dict[int, List[int]] = defaultdict(list)
    
    def add_triple(self, head: str, relation: str, tail: str, chunk_idx: int):
        idx = len(self.triples)
        t = Triple(head.lower().strip(), relation.lower().strip(), 
                   tail.lower().strip(), chunk_idx)
        self.triples.append(t)
        self.entity_index[t.head].append(idx)
        self.entity_index[t.tail].append(idx)
        self.chunk_index[chunk_idx].append(idx)
    
    def get_triples_for_chunk(self, chunk_idx: int) -> List[Triple]:
        return [self.triples[i] for i in self.chunk_index.get(chunk_idx, [])]
    
    def get_triples_for_entity(self, entity: str) -> List[Triple]:
        return [self.triples[i] for i in self.entity_index.get(entity.lower().strip(), [])]
    
    def get_all_entities(self) -> Set[str]:
        return set(self.entity_index.keys())
    
    def get_entities_in_chunk(self, chunk_idx: int) -> Set[str]:
        entities = set()
        for t in self.get_triples_for_chunk(chunk_idx):
            entities.add(t.head)
            entities.add(t.tail)
        return entities
    
    def __len__(self):
        return len(self.triples)

print("KnowledgeGraph class defined.")

KnowledgeGraph class defined.


In [9]:
# Load the LLM for triplet extraction AND answer generation
llm = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.1-8B",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 512,
        "do_sample": False,
    },
    device_map={"": 0}
)
print("LLM loaded.")

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 1971.20it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation 

LLM loaded.


In [10]:
# Triplet extraction prompt (from Figure 3 of the paper)
triplet_extraction_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Extract informative triplets directly from the text following the examples. Do not add any extra words, line breaks, or spaces.

Example 1:
Text: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.
Triplets:
<Scott Derrickson, born in, 1966>
<Scott Derrickson, nationality, America>
<Scott Derrickson, occupation, director>
<Scott Derrickson, occupation, screenwriter>
<Scott Derrickson, occupation, producer>

Example 2:
Text: A Kiss for Corliss is a 1949 American comedy film directed by Richard Wallace and written by Howard Dimsdale.
Triplets:
<A Kiss for Corliss, year, 1949>
<A Kiss for Corliss, country, America>
<A Kiss for Corliss, genre, comedy film>
<A Kiss for Corliss, director, Richard Wallace>
<A Kiss for Corliss, writer, Howard Dimsdale>

Target Text: {text}
Triplets:
"""
)

def parse_triplets(raw_output: str) -> List[Tuple[str, str, str]]:
    """Parse <head, relation, tail> triplets from LLM output."""
    triplets = []
    # Match patterns like <head, relation, tail>
    pattern = r'<([^,]+),\s*([^,]+),\s*([^>]+)>'
    matches = re.findall(pattern, raw_output)
    for h, r, t in matches:
        h, r, t = h.strip(), r.strip(), t.strip()
        if h and r and t:
            triplets.append((h, r, t))
    return triplets

# Test the parser
test = "<Scott Derrickson, born in, 1966>\n<Scott Derrickson, nationality, America>"
print("Parse test:", parse_triplets(test))

Parse test: [('Scott Derrickson', 'born in', '1966'), ('Scott Derrickson', 'nationality', 'America')]


In [11]:
# Build the Knowledge Graph from all chunks
# This is the offline processing step (query-independent, done once)

KG = KnowledgeGraph()

extraction_chain = triplet_extraction_prompt | llm

print(f"Extracting triplets from {len(chunks)} chunks...")
failed_chunks = 0

for idx, chunk in enumerate(chunks):
    try:
        raw_output = extraction_chain.invoke({"text": chunk})
        # Handle different output formats
        if hasattr(raw_output, 'content'):
            text = raw_output.content
        else:
            text = str(raw_output)
        
        # Extract only the triplets portion (after "Triplets:")
        if "Triplets:" in text:
            text = text.split("Triplets:")[-1]
        
        triplets = parse_triplets(text)
        for h, r, t in triplets:
            KG.add_triple(h, r, t, idx)
    except Exception as e:
        failed_chunks += 1
        if failed_chunks <= 3:
            print(f"  Chunk {idx} failed: {e}")

    if (idx + 1) % 20 == 0:
        print(f"  Processed {idx+1}/{len(chunks)} chunks, {len(KG)} triplets so far")

print(f"\nKG Construction Complete:")
print(f"  Total triplets: {len(KG)}")
print(f"  Total entities: {len(KG.get_all_entities())}")
print(f"  Failed chunks: {failed_chunks}/{len(chunks)}")

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Extracting triplets from 21 chunks...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  Processed 20/21 chunks, 425 triplets so far

KG Construction Complete:
  Total triplets: 432
  Total entities: 209
  Failed chunks: 0/21


In [12]:
# Embedding model for semantic retrieval
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda:0"},
    encode_kwargs={"normalize_embeddings": True}
)

# Build FAISS index
vectorstore = FAISS.from_texts(chunks, embeddings)
vectorstore.save_local("faiss_index_kg2rag")

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}  # top-k seed chunks
)

print(f"FAISS index built with {vectorstore.index.ntotal} vectors")

/tmp/ipykernel_2179449/3130995181.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2702.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index built with 21 vectors


In [13]:
def semantic_retrieval(query: str, top_k: int = 10) -> List[int]:
    """
    Semantic-based retrieval (Sec 2.2, Eq. 2).
    Returns indices of top-k seed chunks D_q.
    """
    docs_with_scores = vectorstore.similarity_search_with_score(query, k=top_k)
    
    # Map retrieved docs back to chunk indices
    seed_indices = []
    for doc, score in docs_with_scores:
        text = doc.page_content
        # Find matching chunk index
        for i, chunk in enumerate(chunks):
            if chunk == text and i not in seed_indices:
                seed_indices.append(i)
                break
    
    return seed_indices

# Test
test_seeds = semantic_retrieval(questions[0], top_k=5)
print(f"Query: {questions[0]}")
print(f"Seed chunk indices: {test_seeds}")

Query: In what country is Normandy located?
Seed chunk indices: [4, 1, 20, 5, 6]


In [14]:
def graph_guided_expansion(seed_chunk_indices: List[int], KG: KnowledgeGraph, 
                          m: int = 1) -> Tuple[List[int], List[Triple]]:
    """
    Graph-guided expansion (Sec 2.2, Eq. 3-5).
    
    1. Get subgraph G'_q from seed chunks (Eq. 3)
    2. BFS traverse m-hop neighborhood to get G^m_q (Eq. 4)
    3. Read out expanded chunks D^m_q (Eq. 5)
    
    Args:
        seed_chunk_indices: D_q - indices of seed chunks from semantic retrieval
        KG: The knowledge graph
        m: Number of hops for BFS expansion
    
    Returns:
        expanded_chunk_indices: D^m_q
        expanded_triples: Triples in the expanded subgraph G^m_q
    """
    # Step 1: Get seed entities from seed chunks (Eq. 3)
    seed_entities = set()
    seed_triples = []
    for cidx in seed_chunk_indices:
        seed_entities.update(KG.get_entities_in_chunk(cidx))
        seed_triples.extend(KG.get_triples_for_chunk(cidx))
    
    # Step 2: BFS m-hop expansion (Eq. 4)
    visited_entities = set(seed_entities)
    frontier = set(seed_entities)
    all_triples = list(seed_triples)
    
    for hop in range(m):
        next_frontier = set()
        for entity in frontier:
            for triple in KG.get_triples_for_entity(entity):
                # Add the neighboring entity
                neighbor = triple.tail if triple.head == entity else triple.head
                if neighbor not in visited_entities:
                    next_frontier.add(neighbor)
                    visited_entities.add(neighbor)
                # Collect the triple
                if triple not in all_triples:
                    all_triples.append(triple)
        frontier = next_frontier
    
    # Step 3: Read out expanded chunks (Eq. 5)
    expanded_chunks = set(seed_chunk_indices)
    for triple in all_triples:
        expanded_chunks.add(triple.chunk_idx)
    
    return list(expanded_chunks), all_triples

# Test
test_expanded, test_triples = graph_guided_expansion(test_seeds, KG, m=1)
print(f"Seed chunks: {len(test_seeds)} -> Expanded chunks: {len(test_expanded)}")
print(f"Expanded triples: {len(test_triples)}")

Seed chunks: 5 -> Expanded chunks: 7
Expanded triples: 63


In [15]:
def compute_chunk_similarities(query: str, chunk_indices: List[int]) -> Dict[int, float]:
    """Compute semantic similarity between query and each chunk."""
    query_emb = embeddings.embed_query(query)
    similarities = {}
    for cidx in chunk_indices:
        chunk_emb = embeddings.embed_query(chunks[cidx])
        # Cosine similarity (embeddings are already normalized)
        sim = np.dot(query_emb, chunk_emb)
        similarities[cidx] = float(sim)
    return similarities

def build_undirected_weighted_graph(
    expanded_triples: List[Triple],
    chunk_similarities: Dict[int, float]
) -> Dict[str, Dict[str, List[dict]]]:
    """
    Build undirected weighted graph U^m_q (Eq. 6).
    
    Each edge: (h <-> t, rel: r, src: c, weight: s(q, c))
    """
    graph = defaultdict(lambda: defaultdict(list))
    
    for triple in expanded_triples:
        weight = chunk_similarities.get(triple.chunk_idx, 0.0)
        edge_info = {
            "relation": triple.relation,
            "chunk_idx": triple.chunk_idx,
            "weight": weight
        }
        graph[triple.head][triple.tail].append(edge_info)
        graph[triple.tail][triple.head].append(edge_info)
    
    return graph


def find_connected_components(graph: dict) -> List[Set[str]]:
    """Find connected components B_i in the undirected graph."""
    visited = set()
    components = []
    
    for node in graph:
        if node not in visited:
            component = set()
            queue = [node]
            while queue:
                current = queue.pop(0)
                if current in visited:
                    continue
                visited.add(current)
                component.add(current)
                for neighbor in graph[current]:
                    if neighbor not in visited:
                        queue.append(neighbor)
            components.append(component)
    
    return components


def maximum_spanning_tree(graph: dict, component: Set[str]) -> List[dict]:
    """
    Compute Maximum Spanning Tree (MST) for a connected component (Eq. 7).
    Uses Prim's algorithm with max weight.
    
    Returns list of edges: {head, tail, relation, chunk_idx, weight}
    """
    if len(component) <= 1:
        return []
    
    start = next(iter(component))
    in_tree = {start}
    edges = []
    
    # Priority queue (negate weight for max-heap behavior with min-heap)
    candidates = []
    for neighbor in graph[start]:
        if neighbor in component:
            for edge_info in graph[start][neighbor]:
                heapq.heappush(candidates, (
                    -edge_info["weight"],  # negate for max
                    start, neighbor,
                    edge_info["relation"],
                    edge_info["chunk_idx"]
                ))
    
    while candidates and len(in_tree) < len(component):
        neg_w, u, v, rel, cidx = heapq.heappop(candidates)
        if v in in_tree:
            continue
        
        in_tree.add(v)
        edges.append({
            "head": u, "tail": v,
            "relation": rel, "chunk_idx": cidx,
            "weight": -neg_w
        })
        
        for neighbor in graph[v]:
            if neighbor in component and neighbor not in in_tree:
                for edge_info in graph[v][neighbor]:
                    heapq.heappush(candidates, (
                        -edge_info["weight"],
                        v, neighbor,
                        edge_info["relation"],
                        edge_info["chunk_idx"]
                    ))
    
    return edges


def dfs_chunk_ordering(mst_edges: List[dict]) -> List[int]:
    """
    DFS traversal of MST to get ordered chunk indices.
    Pick the edge with highest weight as root (paper Sec 2.3).
    """
    if not mst_edges:
        return []
    
    # Build adjacency list for the MST
    adj = defaultdict(list)
    for edge in mst_edges:
        adj[edge["head"]].append(edge)
        adj[edge["tail"]].append({
            "head": edge["tail"], "tail": edge["head"],
            "relation": edge["relation"], "chunk_idx": edge["chunk_idx"],
            "weight": edge["weight"]
        })
    
    # Start from the node of the highest-weight edge
    root_edge = max(mst_edges, key=lambda e: e["weight"])
    root = root_edge["head"]
    
    # DFS to collect chunk indices in order
    visited_nodes = set()
    ordered_chunks = []
    
    def dfs(node):
        visited_nodes.add(node)
        for edge in adj[node]:
            target = edge["tail"] if edge["head"] == node else edge["head"]
            if target not in visited_nodes:
                cidx = edge["chunk_idx"]
                if cidx not in ordered_chunks:
                    ordered_chunks.append(cidx)
                dfs(target)
    
    dfs(root)
    return ordered_chunks


def get_triplet_representation(mst_edges: List[dict]) -> str:
    """Get triplet representation of MST for reranking (Eq. 8)."""
    return " ".join([
        f"<{e['head']}, {e['relation']}, {e['tail']}>" 
        for e in mst_edges
    ])

print("Context organization functions defined.")

Context organization functions defined.


In [16]:
def kg2rag_retrieve(query: str, KG: KnowledgeGraph, top_k: int = 10, 
                    m: int = 1) -> str:
    """
    Full KG2RAG retrieval pipeline.
    
    1. Semantic retrieval -> seed chunks D_q
    2. Graph-guided expansion -> expanded chunks D^m_q
    3. KG-based context organization -> filtered & ordered paragraphs
    
    Returns: organized context string
    """
    # Step 1: Semantic-based retrieval
    seed_indices = semantic_retrieval(query, top_k=top_k)
    
    if not seed_indices:
        return ""
    
    # Step 2: Graph-guided expansion (Eq. 3-5)
    expanded_indices, expanded_triples = graph_guided_expansion(
        seed_indices, KG, m=m
    )
    
    if not expanded_triples:
        # Fallback: just use semantic retrieval results
        return "\n\n".join([chunks[i] for i in seed_indices[:top_k]])
    
    # Step 3: KG-based context organization
    # 3a. Compute similarities
    chunk_sims = compute_chunk_similarities(query, expanded_indices)
    
    # 3b. Build undirected weighted graph (Eq. 6)
    ugraph = build_undirected_weighted_graph(expanded_triples, chunk_sims)
    
    # 3c. Find connected components
    components = find_connected_components(ugraph)
    
    # 3d. MST per component (Eq. 7)
    mst_list = []
    for comp in components:
        mst = maximum_spanning_tree(ugraph, comp)
        if mst:
            # Compute relevance score for this MST
            triplet_repr = get_triplet_representation(mst)
            # Use average weight as relevance proxy
            avg_weight = np.mean([e["weight"] for e in mst])
            mst_list.append((avg_weight, mst))
    
    # 3e. Sort MSTs by relevance (descending)
    mst_list.sort(key=lambda x: x[0], reverse=True)
    
    # 3f. DFS ordering and chunk selection until top_k
    organized_chunks = []
    for _, mst in mst_list:
        chunk_order = dfs_chunk_ordering(mst)
        for cidx in chunk_order:
            if cidx not in organized_chunks:
                organized_chunks.append(cidx)
            if len(organized_chunks) >= top_k:
                break
        if len(organized_chunks) >= top_k:
            break
    
    # Add any remaining seed chunks not yet included
    for cidx in seed_indices:
        if cidx not in organized_chunks and len(organized_chunks) < top_k:
            organized_chunks.append(cidx)
    
    # Build final context
    context = "\n\n".join([chunks[i] for i in organized_chunks[:top_k]])
    return context

# Test
test_context = kg2rag_retrieve(questions[0], KG, top_k=5, m=1)
print(f"Query: {questions[0]}")
print(f"Context length: {len(test_context)} chars")
print(f"Preview: {test_context[:500]}...")

Query: In what country is Normandy located?
Context length: 3532 chars
Preview: In the course of the 10th century, the initially destructive incursions of Norse war bands into the rivers of France evolved into more permanent encampments that included local women and personal property. The Duchy of Normandy, which began in 911 as a fiefdom, was established by the treaty of Saint-Clair-sur-Epte between King Charles III of West Francia and the famed Viking ruler Rollo, and was situated in the former Frankish kingdom of Neustria. The treaty offered Rollo and his men the French ...


In [17]:
actor_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a factual assistant whose goal is to produce answers strictly supported by the provided context.

Instructions:
1. Use ONLY the information present in the provided context.
2. Do NOT introduce external knowledge.
3. If the answer cannot be found in the context, respond with: No answer
4. Ensure the answer directly addresses the question.
5. Answer in maximum 3 to 5 words.
6. If the answer is implied, infer it.

Context:
{context}

Question:
{question}

Answer:
"""
)

def actor(question: str, context: str) -> str:
    """Generate answer using LLM with KG2RAG-organized context."""
    actor_chain = actor_prompt | llm
    response = actor_chain.invoke({
        "context": context,
        "question": question
    })
    # Extract answer
    if hasattr(response, 'content'):
        text = response.content
    else:
        text = str(response)
    
    answer = text.split('Answer:')[-1].strip()
    # Clean up - take first line only
    answer = answer.split('\n')[0].strip()
    return answer

# Test
test_answer = actor(questions[0], test_context)
print(f"Q: {questions[0]}")
print(f"A: {test_answer}")
print(f"GT: {ground_truths[0]}")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: In what country is Normandy located?
A: France
GT: France


In [18]:
# Run KG2RAG on all questions
rag_answers = []
retrieved_contexts = []

for i, question in enumerate(questions):
    # KG2RAG retrieval
    context = kg2rag_retrieve(question, KG, top_k=5, m=1)
    retrieved_contexts.append(context)
    
    # Generate answer
    answer = actor(question, context)
    rag_answers.append(answer)
    print(f"Q{i}: {answer}")
    # if (i + 1) % 10 == 0:
    #     print(f"Processed {i+1}/{len(questions)}")

print(f"\nCompleted {len(rag_answers)} answers.")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q0: France


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: The Normans were in Normandy from 911 to 1066.


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q96: William the Conqueror


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q97: 1066


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q98: The ruling class ahead of the Normans was the Frankish aristocracy.
Q99: No answer

Completed 100 answers.


In [20]:
#EM
# all_predictions = []
# all_retrieved_pids = []
# all_retrieved_contexts = []
def normalize(text):
    text = text.strip().lower()
    if text == "unanswerable":
        return ""
    return text

correct = sum([
    normalize(pred) == normalize(gt)
    for pred, gt in zip(rag_answers, ground_truths)
])

total = len(ground_truths)

print("Correct:", correct)
print("Accuracy:", correct / total)

Correct: 16
Accuracy: 0.16


In [22]:
#substring match
def normalize(text):
    text = text.strip().lower()
    if text in ["unanswerable","No answer", "no answer", "no_answer", "n/a"]:
        return ""
    return text

correct = sum([
    normalize(gt) in normalize(pred)
    for pred, gt in zip(rag_answers, ground_truths)
])

print("Accuracy:", correct / len(ground_truths))

Accuracy: 0.76


In [23]:
# --- F1 Score ---
def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()
    
    if not gt_tokens and not pred_tokens:
        return 1.0  # both empty = correct
    if not gt_tokens or not pred_tokens:
        return 0.0
    
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_common = sum(common.values())
    
    if num_common == 0:
        return 0.0
    
    precision = num_common / len(pred_tokens)
    recall = num_common / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

f1_scores = [compute_f1(pred, gt) for pred, gt in zip(rag_answers, ground_truths)]
avg_f1 = np.mean(f1_scores)
print(f"Average F1 Score: {avg_f1:.4f}")

Average F1 Score: 0.2606


In [25]:

dataset_ragas = []

for q, ctx, ans, ref in zip(questions, retrieved_contexts, rag_answers, ground_truths):
    dataset_ragas.append({
        "user_input": q,
        "retrieved_contexts": ctx if isinstance(ctx, list) else [ctx],
        "response": ans,
        "reference": ref
    })

In [26]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset_ragas)

In [27]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from langchain_community.embeddings import OllamaEmbeddings
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness


In [28]:
json_prompt = """
You are a strict JSON output generator used for evaluation in RAG systems.
You must always reply ONLY with a valid JSON object — no explanations, no extra text.

If the question asks for a score, output: {"score": <float between 0 and 1>}
If the question asks for a label (e.g., yes/no), output: {"label": "yes"} or {"label": "no"}
Do not include extra keys, comments, or markdown fences.
"""
langchain_llm= ChatOllama(model="mistral",temperature=0,system=json_prompt)


langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")

/tmp/ipykernel_2179449/241535895.py:9: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  langchain_llm= ChatOllama(model="mistral",temperature=0,system=json_prompt)
/tmp/ipykernel_2179449/241535895.py:12: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")


In [29]:
result = evaluate(dataset=evaluation_dataset,
                  batch_size=2,
                #   metrics=[ LLMContextRecall(), Faithfulness(),FactualCorrectness()],
                  metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
                  llm=langchain_llm,embeddings=langchain_embeddings)
print(result)

Evaluating:  12%|█▏        | 36/300 [02:21<13:50,  3.15s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[37]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating:  13%|█▎        | 38/300 [02:55<31:18,  7.17s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to

{'context_recall': 0.4245, 'faithfulness': 0.5260, 'factual_correctness(mode=f1)': 0.3963}


In [30]:
# Answer similarity using sentence-transformers
from sentence_transformers import SentenceTransformer, util
import torch

qa_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cuda:0")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3948.46it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
# qa_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cuda:0")

sim_scores = []
skipped = 0

for pred, gt in zip(rag_answers, ground_truths):
    norm_pred = normalize(pred)
    norm_gt = normalize(gt)
    
    # Both unanswerable
    if not norm_gt and not norm_pred:
        sim_scores.append(1.0)
        continue
    
    # Skip if ground truth is empty but prediction isn't (or vice versa)
    if not norm_gt:
        skipped += 1
        continue
    
    if not norm_pred:
        sim_scores.append(0.0)
        continue
    
    ref_emb = qa_model.encode(gt, convert_to_tensor=True)
    ans_emb = qa_model.encode(pred, convert_to_tensor=True)
    score = util.cos_sim(ref_emb, ans_emb).item()
    sim_scores.append(score)

print(f"Skipped {skipped} samples")
print(f"Valid samples: {len(sim_scores)}/{len(rag_answers)}")
print(f"Answer Similarity (cosine): {np.mean(sim_scores):.4f}")

Skipped 47 samples
Valid samples: 53/100
Answer Similarity (cosine): 0.5986
